In [ ]:
import time

started = time.perf_counter()

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, T5ForConditionalGeneration


In [ ]:
mode = "beam4"

root = next(Path("/kaggle/input").rglob("test_questions.jsonl")).parent
model_path = root / "model"
questions_path = root / "test_questions.jsonl"
output_dir = Path("/kaggle/working")

batch_size = 256
max_new_tokens = 256
seed = 42
device = "cuda:0"
dtype = torch.float16


In [ ]:
assert mode in ("greedy", "beam4")
rows = [json.loads(line) for line in questions_path.read_text().splitlines() if line.strip()]
questions = [row["question"] for row in rows]
assert len(questions) == 1919

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(
    model_path, torch_dtype=dtype, local_files_only=True,
).to(device).eval()
encoded = tokenizer(questions, truncation=True, max_length=256)
features = [{"input_ids": ids, "attention_mask": mask}
            for ids, mask in zip(encoded["input_ids"], encoded["attention_mask"])]
order = sorted(range(len(rows)), key=lambda i: len(features[i]["input_ids"]), reverse=True)
output_dir.mkdir(parents=True, exist_ok=True)
torch.cuda.synchronize()
setup_seconds = time.perf_counter() - started
torch.cuda.get_device_name(0)


In [ ]:
def generate(beams):
    torch.manual_seed(seed)
    size = max(1, batch_size // beams)
    answers = [None] * len(rows)
    hit_length_limit = 0
    with torch.inference_mode():
        for start in tqdm(range(0, len(order), size)):
            indices = order[start:start + size]
            batch = tokenizer.pad([features[i] for i in indices], padding=True,
                                  pad_to_multiple_of=8, return_tensors="pt").to(device)
            predictions = model.generate(
                **batch, num_beams=beams, num_return_sequences=1, do_sample=False,
                use_cache=True, max_new_tokens=max_new_tokens,
            )[:, 1:].cpu().tolist()
            for index, tokens in zip(indices, predictions):
                finished = tokenizer.eos_token_id in tokens
                if finished:
                    tokens = tokens[:tokens.index(tokenizer.eos_token_id) + 1]
                hit_length_limit += not finished
                answers[index] = tokenizer.decode(tokens, skip_special_tokens=True,
                                                  clean_up_tokenization_spaces=False)
    return answers, hit_length_limit


In [ ]:
torch.cuda.synchronize()
generation_started = time.perf_counter()
answers, hit_length_limit = generate({"greedy": 1, "beam4": 4}[mode])
torch.cuda.synchronize()
generation_seconds = time.perf_counter() - generation_started

output_path = output_dir / f"submission_{mode}.jsonl"
output_path.write_text("".join(json.dumps({"code": code}, ensure_ascii=False) + "\n" for code in answers))
inference_seconds = time.perf_counter() - generation_started
metrics = {"examples": len(rows), "generation_seconds": generation_seconds,
           "inference_seconds": inference_seconds,
           "including_setup_seconds": time.perf_counter() - started,
           "hit_length_limit": hit_length_limit}
(output_dir / "decoding_test.json").write_text(json.dumps({
    "mode": mode, "model": str(model_path), "device": torch.cuda.get_device_name(0),
    "max_new_tokens": max_new_tokens, "hypothesis_batch_size": batch_size,
    "seed": seed, "setup_seconds": setup_seconds, "metrics": {mode: metrics},
}, indent=2))
print(output_path)


In [ ]:
pd.DataFrame({mode: metrics}).T
